# MQTT Flow Evidence

This notebook captures runtime evidence of the MQTT publish/subscribe flow for the Week 4 lab. It complements the integration tests by producing observable artifacts for the lab report.

## Prerequisites

1. Start the local broker:
   ```bash
   make up
   ```
2. Ensure the dashboard is reachable at http://localhost:18083 (user `admin`, password `public`).
3. Run the cells below in order. Each cell appends its output to the evidence collected.

Cells use top-level `await` so they run inside Jupyter's own event loop. Do not call `asyncio.run` from a cell: the kernel already owns the running loop.

In [15]:
import asyncio
import json
from datetime import UTC, datetime
from uuid import uuid4

import aiomqtt

BROKER_HOST = "localhost"
BROKER_PORT = 1883
TOPIC_PREFIX = f"notebook/evidence/{uuid4().hex[:6]}"

## 1. Broker connectivity

Publish a small message to a temporary topic and confirm receipt.

In [16]:
async def publish_and_receive(payload: bytes, qos: int, retain: bool = False):
    topic = f"{TOPIC_PREFIX}/roundtrip"
    received: list[str] = []

    async def subscriber() -> None:
        async with aiomqtt.Client(BROKER_HOST, BROKER_PORT) as client:
            await client.subscribe(topic, qos=qos)
            async with asyncio.timeout(5.0):
                async for message in client.messages:
                    received.append(message.payload.decode())
                    return

    sub_task = asyncio.create_task(subscriber())
    await asyncio.sleep(0.3)
    async with aiomqtt.Client(BROKER_HOST, BROKER_PORT) as pub_client:
        await pub_client.publish(topic, payload, qos=qos, retain=retain)
    await sub_task
    return topic, received


topic, received = await publish_and_receive(b"hello-broker", qos=1)
print(f"Topic: {topic}")
print(f"Received: {received}")

Topic: notebook/evidence/543549/roundtrip
Received: ['hello-broker']


## 2. SensorReading round-trip

Publish a full ``SensorReading`` payload and decode it on the subscriber side to prove the wire contract is stable.

In [17]:
reading = {
    "reading_id": str(uuid4()),
    "device_id": "notebook-sensor-001",
    "temperature_celsius": 22.5,
    "timestamp": datetime.now(tz=UTC).isoformat(),
}
payload = json.dumps(reading, separators=(",", ":")).encode()

topic, received = await publish_and_receive(payload, qos=2)
decoded = json.loads(received[0])
print(json.dumps(decoded, indent=2))

{
  "reading_id": "508699c5-d7b4-4855-947d-dd346a946b8a",
  "device_id": "notebook-sensor-001",
  "temperature_celsius": 22.5,
  "timestamp": "2026-09-23T03:03:35.417562+00:00"
}


## 3. QoS comparison

For each QoS level, publish and confirm receipt.

In [18]:
results = {}
for qos in (0, 1, 2):
    payload = f"qos-{qos}-payload".encode()
    _, received = await publish_and_receive(payload, qos=qos)
    results[qos] = received[0] if received else None

for qos, value in results.items():
    print(f"QoS {qos}: {value}")

QoS 0: qos-0-payload
QoS 1: qos-1-payload
QoS 2: qos-2-payload


## 4. Retained message

Publish a retained message, then attach a fresh subscriber to confirm it receives the message without the publisher being online.

In [19]:
async def publish_retained_then_subscribe() -> str:
    topic = f"{TOPIC_PREFIX}/status"
    async with aiomqtt.Client(BROKER_HOST, BROKER_PORT) as pub:
        await pub.publish(topic, b"online", qos=1, retain=True)
    await asyncio.sleep(0.3)
    async with aiomqtt.Client(BROKER_HOST, BROKER_PORT) as sub:
        await sub.subscribe(topic, qos=1)
        async with asyncio.timeout(3.0):
            async for message in sub.messages:
                return message.payload.decode()
    return "<no message>"


print(f"Retained payload: {await publish_retained_then_subscribe()}")

Retained payload: online


## 5. Observations

Record the observed behavior for the lab report:

- **QoS 0:** the message is delivered but the broker makes no delivery guarantee.
- **QoS 1:** the broker retries until it receives a PUBACK; duplicates are possible.
- **QoS 2:** the four-way handshake guarantees exactly-once delivery, at the cost of higher latency and more round-trips.
- **Retained messages:** allow late subscribers to receive the last known state immediately after subscribing.

Screenshots of the EMQX dashboard (http://localhost:18083) can be captured at each step to complement this evidence.